# README Benchmark Runner

Use the configuration cell to select a phase. Start with `prepare`, then run `generate`, `score`, and `report` separately. The default phase makes no model calls.


## Configuration

In [ ]:
EXPERIMENT = {
    "run_name": "readme_2026_three_models",
    "phase": "prepare",  # prepare, generate, score, or report
    "data_csv": "data/repo_list.csv",
    "tools": {name: {"enabled": True} for name in ("baseline", "osa", "readmeready", "larch", "opencode")},
    "models": ["openai/gpt-6-luna", "anthropic/claude-haiku-5.5", "z-ai/glm-5.3-flash"],
    "judge": {"api": "openrouter", "base_url": "https://openrouter.ai/api/v1", "model": "openai/gpt-6-sol"},
    "reset": {"repos": False, "tool_outputs": False, "evaluation": False},
    "limit_repos": None,
}


## Imports And Paths

In [ ]:
from pathlib import Path
import importlib
import sys

cwd = Path.cwd()
candidates = [cwd, cwd / "benchmark_README", *cwd.parents]
BM = next((p for p in candidates if (p / "src" / "notebook_utils.py").is_file()), cwd)
if str(BM) not in sys.path:
    sys.path.insert(0, str(BM))

import src.notebook_utils as notebook_utils
import src.tool_runners as tool_runners
import src.evaluation as evaluation
import src.reporting as reporting

importlib.reload(notebook_utils)
importlib.reload(tool_runners)
importlib.reload(evaluation)
importlib.reload(reporting)

from src.notebook_utils import build_paths, ensure_directories, load_env_file, read_repo_table, prepare_repositories
from src.tool_runners import run_selected_tools
from src.evaluation import evaluate_outputs
from src.reporting import export_table

load_env_file()
paths = build_paths(EXPERIMENT["run_name"])
ensure_directories(paths)
print(paths)


## Load Repository Table

In [ ]:
repo_df = read_repo_table(EXPERIMENT["data_csv"])
if EXPERIMENT.get("limit_repos"):
    repo_df = repo_df.head(int(EXPERIMENT["limit_repos"]))
repo_df.head()


## Pre-flight: Clone, Checkout, Snapshot

In [ ]:
import pandas as pd
if EXPERIMENT["phase"] == "prepare":
    preflight = prepare_repositories(repo_df, paths, reset=bool(EXPERIMENT["reset"]["repos"]))
else:
    preflight = pd.read_csv(paths.experiment_dir / "preflight_status.csv")
    missing = set(repo_df["repo_slug"]) - set(preflight["repo_slug"])
    if missing:
        raise RuntimeError(f"{len(missing)} repositories have no prepared input")
    failed = preflight[preflight["status"] != "done"]
    if not failed.empty:
        raise RuntimeError(f"{len(failed)} repositories failed preparation")
display(preflight[["repo_slug", "repo_type", "status"]])


## Run Selected Tools

In [ ]:
if EXPERIMENT["phase"] == "generate":
    tool_status = run_selected_tools(repo_df, paths, EXPERIMENT)
    display(tool_status.tail(20))
else:
    print("Generation deferred; set phase to generate in a later run.")


## Evaluate Generated READMEs

In [ ]:
if EXPERIMENT["phase"] == "score":
    eval_rows = evaluate_outputs(paths, EXPERIMENT, reset=bool(EXPERIMENT["reset"]["evaluation"]))
    display(eval_rows.tail(20))
else:
    print("GEval deferred; set phase to score after generation.")


## Summary

In [ ]:
if EXPERIMENT["phase"] == "report":
    result = export_table(repo_df, paths, EXPERIMENT["models"], EXPERIMENT["judge"]["model"])
    display(result["markdown"])
else:
    print("Table export deferred until scoring is complete.")
